In [0]:
from pyspark.sql import functions as F, Window

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

df_info = spark.table("workspace.silver.tb_info_filmes")

window_sk = Window.orderBy("id_filme")

df_dim_movies = df_info.select(
    F.row_number().over(window_sk).alias("sk_movie_id"),
    F.col("id_filme").cast("string").alias("id_filme"),
    "titulo",
    "data_lancamento",
    "ano_lancamento",
    "duracao_minutos",
    "idioma_original",
    "status_filme",
    "sinopse"
)

(df_dim_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("workspace.gold.dim_movies"))

print(f"OK: workspace.gold.dim_movies -> {df_dim_movies.count()} linhas")
display(df_dim_movies.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


OK: workspace.gold.dim_movies -> 97879 linhas


sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
1,14564,Rings,2017-02-01,2017,102,en,Lançado,"Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die."
2,32471,Mixtape,2021-12-03,2021,94,en,Lançado,null
3,38258,Grizzly II: Revenge,2020-02-17,2020,74,en,Lançado,All hell breaks loose when a giant grizzly
4,38492,Billy Joel - Live at Yankee Stadium,2022-06-22,2022,86,en,Lançado,Billy Joel plays his greatest hits in the Big Apple.
5,38700,Bad Boys for Life,2020-01-15,2020,124,en,Lançado,"Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel."
6,42018,The Horse Thief,2019-03-19,2019,88,zh,Lançado,"Devout Buddhists, Norbu and Dolma live with their young son Tashi in a clan in Tibet. Norbu is a highwayman. After Norbu is charged with stealing from the temple, he and his family are banished. Impoverished and marginalized, they can do little when their beloved son becomes ill. Tashi dies of a fever. After a second son is born, Norbu focuses his every action on keeping this child alive, seeking re-admission to the clan for his wife and child, then risking all to save them from isolation and starvation in winter."
7,42330,Monkey Magic,2018-09-22,2018,66,zh,Lançado,"Dearth Voyd, the ruler of the dark side of the universe, wants to turn the human world into a place of violence and evil!! However, a huge meteor falls upon Flower-Fruit mountain giving birth to Kongo, a monkey made of stone! Kongo, now sets out on his quest to fight evil and restore order to a desperate land. Episodes 1-3"
8,43074,Ghostbusters,2016-07-14,2016,117,en,Lançado,"Following a ghost invasion of Manhattan, paranormal enthusiasts Erin Gilbert and Abby Yates, nuclear engineer Jillian Holtzmann, and subway worker Patty Tolan band together to stop the otherworldly threat."
9,45033,20 Seconds of Joy,2018-01-01,2018,60,de,Lançado,"Traces the story of an extreme athlete, past and present; but also explores the psychology behind life, death, risk and the confrontation of fear."
10,46983,The Song of Styrene,2022-05-23,2022,13,fr,Lançado,Le chant du Styrène is a 1958 French documentary film directed by Alain Resnais. The film was an order by French industrial group Pechiney to highlight the merits of plastics.


In [0]:
df_generos = spark.table("workspace.silver.tb_generos")

window_genre = Window.orderBy("nome_genero")

df_dim_genres = (df_generos
    .select("nome_genero")
    .distinct()
    .select(
        F.row_number().over(window_genre).alias("sk_genre_id"),
        "nome_genero"
    ))

(df_dim_genres.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_genres"))

print(f"OK: workspace.gold.dim_genres -> {df_dim_genres.count()} linhas")
display(df_dim_genres)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


OK: workspace.gold.dim_genres -> 19 linhas


sk_genre_id,nome_genero
1,Action
2,Adventure
3,Animation
4,Comedy
5,Crime
6,Documentary
7,Drama
8,Family
9,Fantasy
10,History


In [0]:
df_pessoas_empresas = spark.table("workspace.silver.tb_pessoas_empresas")

window_people = Window.orderBy("nome_pessoa_empresa", "tipo_entidade")

df_dim_people = (df_pessoas_empresas
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select("nome_pessoa_empresa", "tipo_entidade")
    .distinct()
    .select(
        F.row_number().over(window_people).alias("sk_person_id"),
        F.col("nome_pessoa_empresa").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    ))

(df_dim_people.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_people"))

print(f"OK: workspace.gold.dim_people -> {df_dim_people.count()} linhas")

window_companies = Window.orderBy("nome_pessoa_empresa")

df_dim_companies = (df_pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select("nome_pessoa_empresa")
    .distinct()
    .select(
        F.row_number().over(window_companies).alias("sk_company_id"),
        F.col("nome_pessoa_empresa").alias("nome_produtora")
    ))

(df_dim_companies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_companies"))

print(f"OK: workspace.gold.dim_companies -> {df_dim_companies.count()} linhas")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


OK: workspace.gold.dim_people -> 414718 linhas
OK: workspace.gold.dim_companies -> 42290 linhas


In [0]:
df_dim_movies_ref = spark.table("workspace.gold.dim_movies").select("sk_movie_id", "id_filme")

df_dim_genres_ref = spark.table("workspace.gold.dim_genres")

df_bridge_genre = (df_generos
    .join(df_dim_movies_ref, on="id_filme", how="inner")
    .join(df_dim_genres_ref, on="nome_genero", how="inner")
    .select("sk_movie_id", "sk_genre_id")
    .distinct())

(df_bridge_genre.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.bridge_movie_genre"))

print(f"OK: bridge_movie_genre -> {df_bridge_genre.count()} linhas")

df_dim_people_ref = spark.table("workspace.gold.dim_people")

df_bridge_person = (df_pessoas_empresas
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(df_dim_movies_ref, on="id_filme", how="inner")
    .join(
        df_dim_people_ref,
        (df_pessoas_empresas.nome_pessoa_empresa == df_dim_people_ref.nome_pessoa) &
        (df_pessoas_empresas.tipo_entidade == df_dim_people_ref.tipo_pessoa),
        how="inner"
    )
    .select("sk_movie_id", "sk_person_id")
    .distinct())

(df_bridge_person.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.bridge_movie_person"))

print(f"OK: bridge_movie_person -> {df_bridge_person.count()} linhas")

df_dim_companies_ref = spark.table("workspace.gold.dim_companies")

df_bridge_company = (df_pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .join(df_dim_movies_ref, on="id_filme", how="inner")
    .join(
        df_dim_companies_ref,
        df_pessoas_empresas.nome_pessoa_empresa == df_dim_companies_ref.nome_produtora,
        how="inner"
    )
    .select("sk_movie_id", "sk_company_id")
    .distinct())

(df_bridge_company.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.bridge_movie_company"))

print(f"OK: bridge_movie_company -> {df_bridge_company.count()} linhas")

OK: bridge_movie_genre -> 140225 linhas
OK: bridge_movie_person -> 756390 linhas
OK: bridge_movie_company -> 112568 linhas


In [0]:
df_financeiro = spark.table("workspace.silver.tb_financeiro_filmes")
df_metricas = spark.table("workspace.silver.tb_metricas_engajamento")
df_dim_movies_full = spark.table("workspace.gold.dim_movies")
df_movies_lancados = df_dim_movies_full.filter(F.col("status_filme") == "Lançado").select("sk_movie_id", "id_filme")

df_fact = (df_movies_lancados
    .join(df_financeiro, on="id_filme", how="left")
    .join(df_metricas, on="id_filme", how="left")
    .select(
        "sk_movie_id",
        "orcamento_usd", "receita_usd", "lucro_usd",
        "orcamento_brl", "receita_brl", "lucro_brl",
        "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
        "nota_media_imdb", "qtd_votos_imdb"
    ))

(df_fact.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.fact_movies_performance"))

total_fact = df_fact.count()
distintos = df_fact.select("sk_movie_id").distinct().count()
print(f"OK: fact_movies_performance -> {total_fact} linhas | sk_movie_id distintos: {distintos}")

OK: fact_movies_performance -> 96463 linhas | sk_movie_id distintos: 96463


In [0]:
df_avaliacoes = spark.table("workspace.silver.tb_avaliacoes_usuarios")

df_avaliacoes_agg = (df_avaliacoes
    .groupBy("id_filme")
    .agg(
        F.count("*").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).alias("nota_media_usuarios")
    ))

df_dim_movies_ref2 = spark.table("workspace.gold.dim_movies").select("sk_movie_id", "id_filme")
df_dim_reviews_pre = (df_avaliacoes_agg
    .join(df_dim_movies_ref2, on="id_filme", how="inner")
    .select("sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios"))

window_review = Window.orderBy("sk_movie_id")

df_dim_reviews = df_dim_reviews_pre.select(
    F.row_number().over(window_review).alias("sk_review_id"),
    "sk_movie_id",
    "qtd_avaliacoes_usuarios",
    "nota_media_usuarios"
)

(df_dim_reviews.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_reviews"))

print(f"OK: workspace.gold.dim_reviews -> {df_dim_reviews.count()} linhas")
display(df_dim_reviews.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


OK: workspace.gold.dim_reviews -> 27303 linhas


sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
1,2,1,8.3
2,3,2,5.15
3,4,1,2.5
4,5,1,0.1
5,10,1,0.2
6,15,1,4.7
7,16,1,4.9
8,24,1,9.3
9,26,1,1.2
10,28,1,8.2


In [0]:
df_bridge_person_full = spark.table("workspace.gold.bridge_movie_person")
df_dim_people_full = spark.table("workspace.gold.dim_people")

df_pessoas_join = df_bridge_person_full.join(df_dim_people_full, on="sk_person_id", how="inner")

df_atores_agg = (df_pessoas_join
    .filter(F.col("tipo_pessoa") == "Ator")
    .groupBy("sk_movie_id")
    .agg(F.slice(F.collect_list("nome_pessoa"), 1, 3).alias("lista_atores"))
    .withColumn("atores_principais", F.array_join("lista_atores", ", "))
    .select("sk_movie_id", "atores_principais"))

window_diretor = Window.partitionBy("sk_movie_id").orderBy("nome_pessoa")

df_diretores_agg = (df_pessoas_join
    .filter(F.col("tipo_pessoa") == "Diretor")
    .withColumn("rn", F.row_number().over(window_diretor))
    .filter(F.col("rn") == 1)
    .select("sk_movie_id", F.col("nome_pessoa").alias("diretor")))

print(f"Filmes com atores mapeados: {df_atores_agg.count()}")
print(f"Filmes com diretor mapeado: {df_diretores_agg.count()}")

Filmes com atores mapeados: 79627
Filmes com diretor mapeado: 84244


In [0]:
df_movies_ctx = spark.table("workspace.gold.dim_movies")
df_fact_ctx = spark.table("workspace.gold.fact_movies_performance")

df_contexto_base = (df_movies_ctx
    .join(df_fact_ctx, on="sk_movie_id", how="left")
    .join(df_atores_agg, on="sk_movie_id", how="left")
    .join(df_diretores_agg, on="sk_movie_id", how="left"))

df_contexto_tratado = (df_contexto_base
    .withColumn("ano_str", F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("ano não informado")))
    .withColumn("receita_str", F.coalesce(
        F.concat(F.lit("US$ "), F.format_number(F.col("receita_usd"), 0)),
        F.lit("receita não divulgada")
    ))
    .withColumn("orcamento_str", F.coalesce(
        F.concat(F.lit("US$ "), F.format_number(F.col("orcamento_usd"), 0)),
        F.lit("orçamento não divulgado")
    ))
    .withColumn("atores_str", F.coalesce(F.col("atores_principais"), F.lit("elenco não divulgado")))
    .withColumn("diretor_str", F.coalesce(F.col("diretor"), F.lit("um diretor não divulgado")))
    .withColumn("sinopse_str", F.coalesce(
        F.when(F.trim(F.col("sinopse")) == "", None).otherwise(F.col("sinopse")),
        F.lit("sinopse não disponível")
    ))
)

df_contexto_tratado.select(
    "titulo", "ano_str", "receita_str", "orcamento_str", "atores_str", "diretor_str"
).show(5, truncate=False)

+-----------------------------------+-------+---------------------+-----------------------+-------------------------------------------------+------------------------+
|titulo                             |ano_str|receita_str          |orcamento_str          |atores_str                                       |diretor_str             |
+-----------------------------------+-------+---------------------+-----------------------+-------------------------------------------------+------------------------+
|Rings                              |2017   |US$ 83,080,890       |US$ 25,000,000         |Zach Roerig, Laura Slade Wiggins, Johnny Galecki |F. Javier Gutiérrez     |
|Mixtape                            |2021   |receita não divulgada|orçamento não divulgado|Jackson Rathbone, Lucas Yao, Gemma Brooke Allen  |Valerie Weiss           |
|Grizzly II: Revenge                |2020   |receita não divulgada|US$ 7,500,000          |elenco não divulgado                             |um diretor não divulgado

In [0]:
df_gold_context = df_contexto_tratado.select(
    F.col("id_filme").alias("movie_id"),
    "titulo",
    F.concat(
        F.lit("O filme "), F.col("titulo"),
        F.lit(", lançado no ano de "), F.col("ano_str"),
        F.lit(", faturou "), F.col("receita_str"),
        F.lit(" e teve um custo de "), F.col("orcamento_str"),
        F.lit(". Estrelado por "), F.col("atores_str"),
        F.lit(" e dirigido por "), F.col("diretor_str"),
        F.lit(", o filme possui a seguinte sinopse: "), F.col("sinopse_str"),
        F.lit(".")
    ).alias("llm_context_document")
)

(df_gold_context.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.gold_genai_movies_context"))

total = df_gold_context.count()
nulos_doc = df_gold_context.filter(F.col("llm_context_document").isNull()).count()
print(f"OK: gold_genai_movies_context -> {total} linhas | documentos NULL: {nulos_doc}")

df_gold_context.filter(F.col("titulo") == "Grizzly II: Revenge").show(truncate=False)

OK: gold_genai_movies_context -> 97879 linhas | documentos NULL: 0
+--------+-------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|movie_id|titulo             |llm_context_document                                                                                                                                                                                                                                                           |
+--------+-------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|38258   |Grizzly II: Re